INTEGRACION CEM EDAFOLOGIA

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# Instala el motor de lectura rápida (obligatorio para no saturar RAM)
!pip install pyogrio

import geopandas as gpd
import rasterio
from rasterio import features
import numpy as np
import matplotlib.pyplot as plt

path_base = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/Datos/'
path_shp = path_base + 'cdv_edaf_esc_250k_serie II_cont_nac.shp'
path_tiff = path_base + 'CEM_Mosaico_Corregido.tif'
output_path = path_base + 'infiltracion_veracruz_alpha.tif'

try:
    # 1. Obtener solo los metadatos del Raster (Sin cargar la imagen a RAM)
    with rasterio.open(path_tiff) as src:
        raster_meta = src.meta.copy()
        raster_bounds = src.bounds
        raster_transform = src.transform
        raster_shape = src.shape
        raster_crs = src.crs

    print("Metadatos del Raster leídos. Iniciando carga selectiva del Shapefile...")

    # 2. Cargar el Shapefile usando 'pyogrio' y filtrando por columnas y área
    # Esto evita cargar los datos de todo México
    bbox_tuple = (raster_bounds.left, raster_bounds.bottom, raster_bounds.right, raster_bounds.top)

    eda_gdf = gpd.read_file(
        path_shp,
        bbox=bbox_tuple,
        engine='pyogrio',
        columns=['GRUPO1', 'geometry'] # Solo cargamos lo estrictamente necesario
    )

    if eda_gdf.empty:
        raise ValueError("No se encontraron datos de suelo para el área del Raster.")

    print(f"Polígonos cargados: {len(eda_gdf)}. Alineando coordenadas...")

    # 3. Alinear CRS y Reclasificar
    if eda_gdf.crs is None: eda_gdf.crs = raster_crs
    if eda_gdf.crs != raster_crs:
        eda_gdf = eda_gdf.to_crs(raster_crs)

    mapping = {
        'Vertisol': 0.90, 'Feozem': 0.65, 'Regosol': 0.40,
        'Cuerpo de agua': 1.0, 'Zona urbana': 0.95
    }
    eda_gdf['valor_infiltra'] = eda_gdf['GRUPO1'].map(mapping).fillna(0.5)

    # 4. Rasterización eficiente
    print("Iniciando rasterización...")
    geoms = ((geom, val) for geom, val in zip(eda_gdf.geometry, eda_gdf['valor_infiltra']))

    rasterized_soil = features.rasterize(
        geoms,
        out_shape=raster_shape,
        transform=raster_transform,
        fill=0.5,
        dtype='float32'
    )

    # 5. Guardar con compresión LZW para ahorrar espacio en Drive
    raster_meta.update(dtype='float32', count=1, compress='lzw', nodata=0)
    with rasterio.open(output_path, 'w', **raster_meta) as dst:
        dst.write(rasterized_soil, 1)

    print(f"¡Éxito! Archivo guardado en: {output_path}")

    # Visualización minificada para no saturar el buffer del navegador
    plt.figure(figsize=(8, 8))
    plt.imshow(rasterized_soil[::10, ::10], cmap='YlGnBu') # Muestra 1 de cada 10 pixeles
    plt.title("Vista previa (resolución reducida)")
    plt.show()

except Exception as e:
    print(f"Error crítico: {e}")

Metadatos del Raster leídos. Iniciando carga selectiva del Shapefile...
Error crítico: No se encontraron datos de suelo para el área del Raster.


In [9]:
import os

# Listar el contenido de la carpeta path_base
print(f"Contenido de la carpeta {path_base}:")
for item in os.listdir(path_base):
    print(item)


Contenido de la carpeta /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/Datos/:
CEM_Mosaico_Corregido.tif
cdv_edaf_esc_250k_serie II_cont_nac.dbf
cdv_edaf_esc_250k_serie II_cont_nac.shx
cdv_edaf_esc_250k_serie II_cont_nac.shp.xml
cdv_edaf_esc_250k_serie II_cont_nac.shp
metadatos_cdv_edaf_esc_250k_serie II_cont_nac.txt
metadato_cdv_edaf_esc_250k_serie II_cont_nac.txt
metadato_cdv_edaf_esc_250k_serie II_cont_nac.xml
702825224578_1.pdf
